# Lecture: Relationships Between Categorical Features

A frequency table or bar chart describes the distribution of one categorical feature. Many questions ask whether the distribution of an outcome differs across groups. In this lecture, we will use Titanic passenger records and a simulated A/B test to compare two categorical features.


## Learning goals

By the end of this lecture, you should be able to:

- identify a target feature and a comparison feature;
- create a new categorical feature from a quantitative count;
- create a cross-tabulation of two categorical features;
- distinguish counts, joint percentages, and conditional percentages;
- explain why percentages provide a fairer comparison when groups have different sizes;
- visualize conditional percentages with a bar chart; and
- state a conclusion and limitation supported by the analysis.


## Import the libraries

We use Pandas to read and organize the data and Matplotlib to label and refine the figures.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


# Part 1: Survival of Titanic Passengers

RMS *Titanic* sank in the North Atlantic in April 1912. The passenger dataset contains one row for each of 891 passengers. It includes personal and travel information such as passenger name, age, sex, passenger class, family members aboard, port of departure, fare, and survival.

This is a passenger-only dataset. It does not include the ship's crew.


## Data dictionary

| Feature | Description |
| --- | --- |
| `PassengerId` | Unique passenger identifier |
| `Survived` | Survival outcome: 0 = did not survive, 1 = survived |
| `Pclass` | Passenger class: 1 = first, 2 = second, 3 = third |
| `Name` | Passenger name |
| `Sex` | Sex recorded as male or female |
| `Age` | Age in years |
| `SibSp` | Number of siblings or spouses aboard |
| `Parch` | Number of parents or children aboard |
| `Ticket` | Ticket number |
| `Fare` | Passenger fare |
| `Cabin` | Cabin number, when recorded |
| `Embarked` | Port of departure: C = Cherbourg, Q = Queenstown, S = Southampton |

Historical categories and missing values reflect the surviving records and the way this commonly used dataset was prepared.


## Discussion: predict the target feature

1. Which feature should be the target?
2. What does the target feature represent?


### Discussion notes

`Survived` is the target because it records the outcome we want to understand. It indicates whether each passenger survived.


## The reproducible analysis workflow

1. **Question:** What do we want to learn?
2. **Data:** Which observations and features can help answer it?
3. **Operation:** What should the code select, calculate, or compare?
4. **Check:** Does the result have the expected rows, columns, units, and possible values?
5. **Evidence:** Which result directly answers the question?
6. **Conclusion:** What claim is supported?
7. **Limitation:** What should we avoid concluding?


# Research question 1: Who was more likely to survive?


## Step 1 — Question

> **Who was more likely to survive: male or female passengers aboard the Titanic?**

The target feature is `Survived`. The comparison feature is `Sex`.

### Discussion: predict the result

1. Which group do you predict had the higher survival percentage?
2. What historical assumptions or prior knowledge influenced your prediction?


## Step 2 — Data

Load the local passenger file and inspect the DataFrame. These checks use the same techniques introduced earlier in the course.


In [ ]:
titanic_df = pd.read_csv("data/titanic_passengers.csv")
titanic_df.head()


In [ ]:
titanic_df.tail()


In [ ]:
titanic_df.info()


In [ ]:
titanic_df.isnull().sum()


### Discussion: check the data

1. What does each row represent?
2. Which features are needed for the first research question?
3. Are either of those features missing values?
4. Which other features have missing values?


### Discussion notes

Each row represents one passenger. The first question requires `Sex` and `Survived`, and neither has missing values. `Age`, `Cabin`, and `Embarked` contain missing values.


## Step 3 — Operation

Begin by examining the distribution of each categorical feature. `.value_counts()` reports the frequency of every category. Adding `normalize=True` divides each frequency by the total, and multiplying by 100 converts the proportions to percentages.


In [ ]:
print(titanic_df["Sex"].value_counts())
print(titanic_df["Sex"].value_counts(normalize=True).mul(100).round(1))


In [ ]:
print(titanic_df["Survived"].value_counts())
print(titanic_df["Survived"].value_counts(normalize=True).mul(100).round(1))


A **cross-tabulation**, or cross-tab, displays the combinations of two categorical features. This count table is the **joint distribution** of `Sex` and `Survived`.


In [ ]:
survival_counts = pd.crosstab(
    titanic_df["Sex"],
    titanic_df["Survived"]
)

survival_counts


Counts do not create a fair comparison when the groups contain different numbers of passengers. The question asks for the survival percentage **within each sex**, so each sex must have its own denominator.

`normalize="index"` divides every cell by its row total. Multiplying by 100 makes each row add to 100%.


In [ ]:
survival_percent = (
    pd.crosstab(
        titanic_df["Sex"],
        titanic_df["Survived"],
        normalize="index"
    )
    .mul(100)
    .round(1)
)

survival_percent


### Discussion: counts or percentages?

1. Are the male and female groups the same size?
2. Why would comparing only the number of survivors be unfair?
3. In the percentage table, what population is used as the denominator for each row?


### Discussion notes

The groups are not the same size: there are more male than female passengers. A count can be larger simply because a group began with more people. Row percentages compare survival relative to the number of passengers in each sex, so they answer who was more likely to survive.


## Step 4 — Check

Verify that the count-table row totals equal the number of passengers in each sex and that every row of the conditional percentage table adds to 100%.


In [ ]:
print(survival_counts.sum(axis=1))
print(titanic_df["Sex"].value_counts())
print(survival_percent.sum(axis=1))


## Step 5 — Evidence

Rename the coded survival outcomes so the legend is understandable, then plot the conditional percentages. The height of each full bar is 100%, so the colored sections show the survival distribution within each sex.


In [ ]:
survival_plot = survival_percent.rename(
    columns={0: "Did not survive", 1: "Survived"}
)

survival_plot.plot(
    kind="bar",
    stacked=True,
    color=["steelblue", "darkorange"]
)
plt.title("Titanic Passenger Survival by Sex")
plt.xlabel("Sex")
plt.ylabel("Percent of passengers")
plt.xticks(rotation=0)
plt.legend(title="Survival")
plt.show()


### Discussion: interpret the evidence

1. Which group has the larger `Survived` section?
2. Approximately what percentage of each group survived?
3. Does the figure support your prediction?


### Discussion notes

About 74.2% of female passengers survived, compared with about 18.9% of male passengers. Female passengers therefore had the higher survival percentage in this dataset.


## Step 6 — Conclusion

In this passenger dataset, female passengers were more likely to survive than male passengers. The difference is large: roughly three-quarters of female passengers survived, compared with fewer than one-fifth of male passengers.


## Step 7 — Limitation

This is an observational historical dataset, not a randomized experiment. Sex was related to survival, but it was also related to decisions about evacuation and may have been related to age, passenger class, cabin location, and access to lifeboats. The comparison does not establish that sex alone caused the difference.


# Research question 2: Traveling with parents or children


## Step 1 — Question

> **Were passengers traveling with parents or children more likely to survive than passengers traveling without parents or children?**

`Parch` is a count, but the question asks about two groups. We will create a categorical comparison feature before calculating survival percentages.

### Discussion: predict the result

1. Which group do you predict had the higher survival percentage?
2. Why might traveling with parents or children be related to survival?


## Step 2 — Data

The relevant features are `Parch` and `Survived`. Because the DataFrame is already loaded, we only need to inspect those columns rather than repeat all the general checks.


In [ ]:
titanic_df[["Parch", "Survived"]].head(10)


In [ ]:
print(titanic_df[["Parch", "Survived"]].isnull().sum())
print(titanic_df["Parch"].value_counts().sort_index())


## Step 3 — Operation: create a comparison feature

First, assign every passenger the label `Without parents/children`. Then use `.loc[]` to replace that label only in rows where `Parch` is greater than zero.

`.loc[rows, column]` selects the rows that meet a condition and the column that should be changed. Here, it lets us create readable group labels without changing the original `Parch` values.


In [ ]:
titanic_df["parents_children_status"] = "Without parents/children"

titanic_df.loc[
    titanic_df["Parch"] > 0,
    "parents_children_status"
] = "With parents/children"


Check the calculated feature before using it. We expect every passenger with `Parch` equal to 0 to be in one group and every passenger with a positive value to be in the other.


In [ ]:
pd.crosstab(
    titanic_df["Parch"],
    titanic_df["parents_children_status"]
)


Now calculate the count table and row-conditional survival percentages.


In [ ]:
family_survival_counts = pd.crosstab(
    titanic_df["parents_children_status"],
    titanic_df["Survived"]
)

family_survival_counts


In [ ]:
family_survival_percent = (
    pd.crosstab(
        titanic_df["parents_children_status"],
        titanic_df["Survived"],
        normalize="index"
    )
    .mul(100)
    .round(1)
)

family_survival_percent


## Step 4 — Check

Confirm that the two new categories account for all passengers and that the percentage rows add to 100%.


In [ ]:
print(titanic_df["parents_children_status"].value_counts())
print("Passengers represented:", family_survival_counts.to_numpy().sum())
print(family_survival_percent.sum(axis=1))


## Step 5 — Evidence


In [ ]:
family_survival_plot = family_survival_percent.rename(
    columns={0: "Did not survive", 1: "Survived"}
)

family_survival_plot.plot(
    kind="bar",
    stacked=True,
    color=["steelblue", "darkorange"]
)
plt.title("Titanic Passenger Survival and Parents or Children Aboard")
plt.xlabel("Travel status")
plt.ylabel("Percent of passengers")
plt.xticks(rotation=0)
plt.legend(title="Survival")
plt.show()


### Discussion: interpret the evidence

1. Which group had the higher survival percentage?
2. Approximately how large is the difference?
3. Does the figure support your prediction?


### Discussion notes

About 51.2% of passengers traveling with at least one parent or child survived, compared with about 34.4% of passengers traveling without parents or children. The difference is approximately 16.8 percentage points.


## Step 6 — Conclusion

In this dataset, passengers recorded as traveling with at least one parent or child had a higher survival percentage than passengers recorded as traveling without parents or children.


## Step 7 — Limitation

`Parch` records only the number of parents or children aboard. The dataset documentation notes that some children who traveled with a nanny have `Parch` equal to 0. The feature also does not describe every family relationship, who stayed together during the evacuation, or whether a passenger was traveling with other companions. The groups may differ in age, class, and other features related to survival. This relationship should not be interpreted as proof that traveling with parents or children caused survival.


# Part 2: A/B Testing

An **A/B test** randomly assigns participants to one of two versions of a product, message, or experience. The outcome is then compared across the two experimental conditions.

Random assignment is what makes an A/B test different from the Titanic comparisons. When the groups are created randomly and the experiment is conducted well, a difference in the response can provide evidence that the tested version affected the outcome.


## Example: a museum membership page

A science museum wants more website visitors to begin a free membership. It randomly assigns visitors to one of two pages:

- `Standard page`: the existing page with several membership choices;
- `Simplified page`: a shorter page with one prominent free-membership button.

The response feature `Signed_up` records whether each visitor completed the free-membership signup.

This is a **simulated teaching dataset**. The observations were created to make the structure and interpretation of an A/B test easy to see; they are not evidence about a real museum or website.


## Step 1 — Question

> **Did visitors shown the simplified page sign up more often than visitors shown the standard page?**

### Discussion: identify the features and predict

1. Which feature is the experimental condition?
2. Which feature is the response?
3. Which page do you predict will have the higher signup percentage?


### Discussion notes

`Page_version` is the experimental condition, and `Signed_up` is the response. A prediction should be recorded before examining the outcome.


## Step 2 — Data


In [ ]:
signup_df = pd.read_csv("data/museum_signup_ab.csv")
signup_df.head()


In [ ]:
signup_df.info()


In [ ]:
signup_df.isnull().sum()


## Step 3 — Operation

Inspect the condition and response frequencies, then calculate counts and signup percentages within each page version.


In [ ]:
print(signup_df["Page_version"].value_counts())
print(signup_df["Signed_up"].value_counts())


In [ ]:
signup_counts = pd.crosstab(
    signup_df["Page_version"],
    signup_df["Signed_up"]
)

signup_counts


In [ ]:
signup_percent = (
    pd.crosstab(
        signup_df["Page_version"],
        signup_df["Signed_up"],
        normalize="index"
    )
    .mul(100)
    .round(1)
)

signup_percent


## Step 4 — Check

Check that both experimental conditions contain the expected number of visitors and that each percentage row adds to 100%.


In [ ]:
print(signup_counts.sum(axis=1))
print(signup_percent.sum(axis=1))


## Step 5 — Evidence


In [ ]:
signup_plot = signup_percent.rename(
    columns={"No": "Did not sign up", "Yes": "Signed up"}
)

signup_plot.plot(
    kind="bar",
    stacked=True,
    color=["slategray", "mediumseagreen"]
)
plt.title("Free-Membership Signup by Page Version")
plt.xlabel("Page version")
plt.ylabel("Percent of visitors")
plt.xticks(rotation=0)
plt.legend(title="Signup response")
plt.show()


### Discussion: interpret the A/B test

1. What percentage of visitors signed up after seeing each page?
2. What is the difference in percentage points?
3. Is the difference easy to see in the figure?
4. Does the result support your prediction?


### Discussion notes

In the simulated data, 65% of visitors assigned to the simplified page signed up, compared with 30% assigned to the standard page. That is a difference of 35 percentage points, so the response difference is visually striking.


## Step 6 — Conclusion

In this simulated experiment, visitors assigned to the simplified page signed up substantially more often than visitors assigned to the standard page.


## Step 7 — Limitation

Because this dataset was simulated for teaching, the result cannot support a real product decision. In a real A/B test, we would also verify that random assignment and data collection worked as intended and use statistical inference to assess uncertainty before deciding whether to change the website.


## Procedure for comparing categorical features

1. Identify the target or response feature and the comparison feature.
2. Inspect the frequency of each feature.
3. Create a count cross-tab.
4. Check group totals.
5. Identify the denominator implied by the research question.
6. Calculate the joint or conditional percentages required by the question.
7. Check that the expected percentages add to 100%.
8. Choose a figure that makes the comparison clear.
9. State a conclusion and limitation.


# Optional historical interlude: Charles Joseph Minard and the flow map

Charles Joseph Minard (1781–1870) was a French civil engineer who pioneered the use of flow maps. His best-known visualization, published in 1869, depicts the losses suffered by Napoleon's army during the 1812 campaign in Russia.

Minard combined several features in one figure:

- geographic location and the army's direction of travel;
- the number of troops, represented by the width of the band;
- the advance toward Moscow in tan and the retreat in black; and
- dates and temperatures during the retreat.

The narrowing bands make the army's losses visible without requiring the reader to inspect every printed number.


<img src="images/minard_napoleon_1869.png" alt="Charles Joseph Minard's 1869 flow map of Napoleon's 1812 campaign in Russia" width="920">

*Minard's 1869 flow map of Napoleon's 1812 Russian campaign. [Original image from Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Minard.png), public domain.*


### Discussion

1. What question does Minard's visualization help answer?
2. Which feature is represented by the width of the band?
3. What conclusion becomes visible as the band narrows?
4. What information or limitations would you want before using the figure as historical evidence?


## Key takeaways

- The target or response feature is the outcome we want to understand.
- A cross-tab describes the joint distribution of two categorical features.
- Counts describe how many observations appear in each combination of categories.
- Conditional percentages describe the response within each comparison group.
- `normalize="index"` makes every row add to 100%.
- A calculated categorical feature can turn a quantitative count into groups that match a research question.
- Random assignment distinguishes an experiment from an observational comparison.
- An observed relationship should be reported with a limitation and should not automatically be interpreted as causal.
